# Module E, Problem Set: The Hartree-Fock SCF Engine
## Question 1: Building Helium (2G vs. 4G)

**Student Name:** [TYPE YOUR NAME HERE]  
**Date:** [TYPE DATE HERE]

### Objectives
1. Construct a working Self-Consistent Field (SCF) loop for the Helium atom.
2. Conceptually identify the roles of the Overlap, Fock, and Density matrices.
3. Compare a Double-Zeta (2G) basis set to a Quadruple-Zeta (4G) basis set.
4. Observe the Variational Theorem and Electron Correlation in action.

In [ ]:
# --- RUN THIS CELL FIRST --- 
import numpy as np
import scipy.linalg as la
from scipy.linalg import fractional_matrix_power

np.set_printoptions(precision=5, suppress=True)

# Helper functions for exact s-type atomic integrals (Do not modify)
def overlap(a, b):
    return (2 * np.sqrt(a * b) / (a + b))**1.5
def kinetic(a, b):
    return 3 * a * b / (a + b) * overlap(a, b)
def nuclear(a, b, Z):
    return -Z * (2 * np.sqrt(a + b) / np.sqrt(np.pi)) * overlap(a, b)
def repulsion(a, b, c, d):
    p = a + b; q = c + d
    return (16 * (a * b * c * d)**0.75) / (np.sqrt(np.pi) * p * q * np.sqrt(p + q))

print("Environment and Integral Engine ready!")

---
## Part A: The 2-Gaussian (2G) Baseline

We will start by modeling the Helium atom ($Z=2$) using a Double-Zeta basis set. This means we are giving the computer two $s$-type Gaussian functions to mix together: one tight (large $\alpha$), and one diffuse (small $\alpha$). 

Run the cell below. It calculates all the fixed spatial integrals ($\mathbf{S}$, $\mathbf{h}_{core}$, $\mathbf{V}_{ee}$) before the loop begins, and then executes the 9-Step SCF algorithm. Read through the printed output and the code carefully.


In [ ]:
# 1. Setup
alphas_2G = np.array([0.85, 3.53]) 
Z = 2
N = len(alphas_2G)

# 2. Build Pre-Loop Matrices
S = np.zeros((N, N)); h_core = np.zeros((N, N)); V_ee = np.zeros((N, N, N, N))
for r in range(N):
    for s in range(N):
        a, b = alphas_2G[r], alphas_2G[s]
        S[r, s] = overlap(a, b)
        h_core[r, s] = kinetic(a, b) + nuclear(a, b, Z)
        for t in range(N):
            for u in range(N):
                V_ee[r, s, t, u] = repulsion(a, b, alphas_2G[t], alphas_2G[u])

X = fractional_matrix_power(S, -0.5)

# 3. The SCF Loop
P = np.zeros((N, N)) # Initial Guess: No electron repulsion
energy_old = 0.0

print("Iter | Electronic Energy (Hartrees) | Delta E")
print("-----------------------------------------------")

for iteration in range(25):
    # Build Fock Matrix
    F = np.zeros((N, N))
    for r in range(N):
        for s in range(N):
            F[r, s] = h_core[r, s] 
            for t in range(N):
                for u in range(N):
                    F[r, s] += P[t, u] * (V_ee[r, s, t, u] - 0.5 * V_ee[r, u, t, s])
                    
    # Solve
    F_prime = np.dot(X, np.dot(F, X))
    E, C_prime = la.eigh(F_prime)
    C = np.dot(X, C_prime)
    
    # Rebuild Density Matrix (Helium has 2e-, so we only use the first column of C)
    P_new = np.zeros((N, N))
    for t in range(N):
        for u in range(N):
            P_new[t, u] = 2 * C[t, 0] * C[u, 0]
            
    # Check Energy
    E_elec = 0.5 * np.sum(P_new * (h_core + F))
    delta_E = abs(E_elec - energy_old)
    print(f"{iteration+1:4d} | {E_elec:27.6f} | {delta_E:.2e}")
    
    if delta_E < 1e-6: 
        break
    P = P_new; energy_old = E_elec

print(f"\nFinal 2G Energy: {E_elec:.6f} Hartrees")

---
### Conceptual Intermission

Before we upgrade our basis set, let's make sure we understand the physical math happening in the loop above.

**Question 1.1:** Look at the initial guess for the Density Matrix ($\mathbf{P}$). We set it to a matrix of pure zeros. Mathematically, look at the equation for the Fock Matrix ($\mathbf{F}$). If $\mathbf{P} = 0$, what does $\mathbf{F}$ exactly equal during the very first cycle? What does this mean physically about our initial assumption of the electrons?

**[DOUBLE CLICK THIS CELL TO TYPE YOUR ANSWER]**

**Question 1.2:** We use `fractional_matrix_power` to calculate $\mathbf{X} = \mathbf{S}^{-1/2}$. Why do standard Python eigenvalue solvers (like `la.eigh`) require us to use this transformation matrix instead of just solving the raw $\mathbf{F}$ matrix directly? *(Hint: Think about what the Overlap matrix $\mathbf{S}$ means physically for our Gaussians).*

**[DOUBLE CLICK THIS CELL TO TYPE YOUR ANSWER]**

---
## Part B: Upgrading to 4-Gaussian (4G)

The true experimental ground state energy of Helium is **-2.9033 Hartrees**. Our 2G basis set got us close, but because of the **Variational Theorem**, we know that if we give the computer more flexibility, it will find a lower, more accurate energy.

Let's expand our basis set from a Double-Zeta (2G) to a Quadruple-Zeta (4G) basis. By including extremely tight Gaussians (large $\alpha$) and very diffuse Gaussians (small $\alpha$), the algorithm can mix them together to fake the sharp nuclear cusp and the long tail of a true orbital.

**Your Task:**
Copy the entire 2G SCF code from **Cell 4** and paste it into the cell below. 
Modify it to use the new `alphas_4G` array provided. 
*(Hint: You do not need to change any of the `for` loops! Because we used `N = len(alphas)`, the exact same code will automatically scale to solve a $4 \times 4$ matrix!)*


In [ ]:
alphas_4G = np.array([0.298073, 1.242567, 5.782948, 38.47497])

# YOUR CODE HERE: Paste and modify the SCF engine to run the 4G basis set!



---
### Final Analysis

**Question 1.3:** Compare your converged 4G energy to your 2G energy. Did it go down? Does this obey the Variational Theorem?

**[DOUBLE CLICK THIS CELL TO TYPE YOUR ANSWER]**

**Question 1.4:** Your 4G energy should be approximately `-2.861` Hartrees. Even with four highly optimized Gaussians, we did not hit the true experimental energy of `-2.903` Hartrees. This remaining error ($\sim 0.04$ Hartrees) has a specific name in computational chemistry. What is it called, and what specific physical interaction are we missing by using the "Mean-Field" approximation?

**[DOUBLE CLICK THIS CELL TO TYPE YOUR ANSWER]**